In [2]:
# -*- coding: utf-8 -*-
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or
# implied.
# See the License for the specific language governing permissions and
# limitations under the License.


# Imports

In [3]:
from zenodo_get import download
import os
from lxml import etree
import zipfile
import pandas as pd

# Constant

In [4]:
DATASETS_DIR = os.path.join(os.getcwd(), "datasets")

# Dataset 100 tweets per author

Pan clef dataset from author profiling task

https://zenodo.org/records/3692340

https://doi.org/10.5281/zenodo.3692340

In [5]:
download(record_or_doi="3692340", output_dir=DATASETS_DIR)

2026-03-10 21:47:08.777 | INFO     | zenodo_get.zget:_zenodo_download_logic:306 - Output directory: /home/user/study/4course/who_wrote_it_nlp/notebooks/datasets
2026-03-10 21:47:09.300 | INFO     | zenodo_get.zget:_zenodo_download_logic:404 - Title: PAN19 Author Profiling:  Bots and Gender Profiling
2026-03-10 21:47:09.301 | INFO     | zenodo_get.zget:_zenodo_download_logic:416 - Total size: 129.9 MB
2026-03-10 21:47:09.303 | INFO     | zenodo_get.zget:_zenodo_download_logic:417 - Number of files: 4
2026-03-10 21:47:20.498 | SUCCESS  | zenodo_get.zget:_zenodo_download_logic:461 - All specified files have been processed.


Remove not used parts

In [6]:
unused_files_path = [
    "pan19-author-profiling-earlybirds-20190320.zip",
    "pan19-author-profiling-earlybirds-20190320.zip",
    "pan19-author-profiling-20200229.zip"
]

In [7]:
for file in unused_files_path:
    file_path = os.path.join(DATASETS_DIR, file)
    if os.path.exists(file_path):
        if os.path.isfile(file_path):
            os.remove(file_path)
        else:
            os.rmdir(file_path)

In [8]:
files_for_extract = [
    "pan19-author-profiling-training-dataset-2019-02-18.zip",
    "pan19-author-profiling-test-2019-04-29.zip"
]

In [9]:
for file in files_for_extract:
    file_path = os.path.join(DATASETS_DIR, file)
    if os.path.exists(file_path) and zipfile.is_zipfile(file_path):
        with zipfile.ZipFile(file_path, 'r') as zip_ref:
            zip_ref.extractall(DATASETS_DIR)
        os.remove(file_path)

In [10]:
training_path = os.path.join(DATASETS_DIR, "pan19-author-profiling-training-2019-02-18", "en")
if not os.path.exists(training_path):
    raise FileNotFoundError(f"Training path not found: {training_path}")
test_path = os.path.join(DATASETS_DIR, "pan19-author-profiling-test-2019-04-29", "en")
if not os.path.exists(test_path):
    raise FileNotFoundError(f"Test path not found: {test_path}")
test_truth_path = os.path.join(DATASETS_DIR, "pan19-author-profiling-test-2019-04-29", "en.txt")
if not os.path.exists(test_truth_path):
    raise FileNotFoundError(f"Test truth path not found: {test_truth_path}")

In [18]:
train_df = pd.DataFrame()
for file in os.listdir(training_path):
    if not file.endswith(".xml"):
        continue
    try:
        tmp_df = pd.read_xml(os.path.join(training_path, file))
        tmp_df["id"] = file.split(".")[0]
        train_df = pd.concat([train_df, tmp_df], ignore_index=True)
    except ValueError as e:
        print(f"Error reading file {file}: {e}")

truth_train_df = pd.read_csv(os.path.join(training_path, "truth-train.txt"), sep=":::", header=None, names=["id", "who","gender_label"], engine="python")
train_df = train_df.merge(truth_train_df, on="id", how="left")
train_df.head()

,document,id,who,gender_label
0,RT @FMoniteau: Here are Republican Tax Bracket...,ce2dd62422f31023469e9d8f55f5df1f,human,male
1,Excited to join the rumble today.,a9e2935dda82fb947f51bf24d7f28782,NaN,NaN
2,"RT @mattgallowaycbc: “Canadians: we're polite,...",ef441cb32dd3f524fd2f55e8fa381334,NaN,NaN
3,@emmaguns Brilliant!! Yet again - really loved...,77fd4e38f2f4061d3bb6f7efe82e30a2,human,female
4,RT @TechnicallyRon: Brexit is such a shit idea...,f8ce69192847c64db8e1b322c1676cfd,human,male


In [ ]:
test_df = pd.DataFrame()
for file in os.listdir(test_path):
    if not file.endswith(".xml"):
        continue
    try:
        tmp_df['id'] = file.split(".")[0]
        tmp_df = pd.read_xml(os.path.join(test_path, file))
        test_df = pd.concat([test_df, tmp_df], ignore_index=True)
    except ValueError as e:
        print(f"Error reading file {file}: {e}")

truth_test_df = pd.read_csv(os.path.join(training_path, "truth.txt"), sep=":::", header=None, names=["id", "who","gender_label"], engine="python")
test_df = test_df.merge(truth_test_df, on="id", how="left")
test_df.head()

,document,id,who_x,gender_label_x,who_y,gender_label_y
0,RT @FMoniteau: Here are Republican Tax Bracket...,ce2dd62422f31023469e9d8f55f5df1f,human,male,human,male
1,Excited to join the rumble today.,a9e2935dda82fb947f51bf24d7f28782,NaN,NaN,human,male
2,"RT @mattgallowaycbc: “Canadians: we're polite,...",ef441cb32dd3f524fd2f55e8fa381334,NaN,NaN,human,female
3,@emmaguns Brilliant!! Yet again - really loved...,77fd4e38f2f4061d3bb6f7efe82e30a2,human,female,human,female
4,RT @TechnicallyRon: Brexit is such a shit idea...,f8ce69192847c64db8e1b322c1676cfd,human,male,human,male


# Pipeline

# Baseline params from PAN-CLEF

# Training model on 100 tweets per author

# Importing 1 tweet per author model

# Comparison of 1 tweet vs 100 tweets per author models